In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

In [2]:
def one_bit_adder_subtractor(a, b, carry_borrow_in, mode):
    """
    One-bit full adder-cum-subtractor.

    mode = 0 -> addition
    mode = 1 -> subtraction

    For addition:
        result = A + B + carry_in

    For subtraction:
        result = A - B - borrow_in
    """

    if mode == 0:
        total = a + b + carry_borrow_in

        result = total % 2
        carry_borrow_out = total // 2

    else:
        total = a - b - carry_borrow_in

        if total < 0:
            result = total + 2
            carry_borrow_out = 1
        else:
            result = total
            carry_borrow_out = 0

    return result, carry_borrow_out

In [3]:
print("A B Cin/Bin Mode | Result | Carry/Borrow")
print("-------------------------------------------")

for mode in [0, 1]:

    for a in [0, 1]:

        for b in [0, 1]:

            for carry_borrow_in in [0, 1]:

                result, output = one_bit_adder_subtractor(
                    a,
                    b,
                    carry_borrow_in,
                    mode
                )

                operation = "ADD" if mode == 0 else "SUB"

                print(
                    f"{a} {b}      {carry_borrow_in}      {mode}   |"
                    f"   {result}   |      {output}"
                )

A B Cin/Bin Mode | Result | Carry/Borrow
-------------------------------------------
0 0      0      0   |   0   |      0
0 0      1      0   |   1   |      0
0 1      0      0   |   1   |      0
0 1      1      0   |   0   |      1
1 0      0      0   |   1   |      0
1 0      1      0   |   0   |      1
1 1      0      0   |   0   |      1
1 1      1      0   |   1   |      1
0 0      0      1   |   0   |      0
0 0      1      1   |   1   |      1
0 1      0      1   |   1   |      1
0 1      1      1   |   0   |      1
1 0      0      1   |   1   |      0
1 0      1      1   |   0   |      0
1 1      0      1   |   0   |      0
1 1      1      1   |   1   |      1


In [4]:
def eight_bit_adder_subtractor(a, b, mode):
    """
    8-bit ripple-style reference model.

    mode = 0 -> A + B
    mode = 1 -> A - B

    Returns:
        result, final carry/borrow
    """

    carry_borrow = 0
    result = 0

    for bit in range(8):

        a_bit = (a >> bit) & 1
        b_bit = (b >> bit) & 1

        result_bit, carry_borrow = one_bit_adder_subtractor(
            a_bit,
            b_bit,
            carry_borrow,
            mode
        )

        result |= (result_bit << bit)

    return result, carry_borrow

In [5]:
test_values = [
    (10, 5),
    (25, 12),
    (100, 50),
    (150, 75),
    (200, 25)
]

print("===== 8-BIT ADDITION =====")

for a, b in test_values:

    result, carry = eight_bit_adder_subtractor(
        a,
        b,
        0
    )

    print(
        f"{a} + {b} = {result}"
        f" | Carry = {carry}"
    )

===== 8-BIT ADDITION =====
10 + 5 = 15 | Carry = 0
25 + 12 = 37 | Carry = 0
100 + 50 = 150 | Carry = 0
150 + 75 = 225 | Carry = 0
200 + 25 = 225 | Carry = 0


In [6]:
print("===== 8-BIT SUBTRACTION =====")

for a, b in test_values:

    result, borrow = eight_bit_adder_subtractor(
        a,
        b,
        1
    )

    print(
        f"{a} - {b} = {result}"
        f" | Borrow = {borrow}"
    )

===== 8-BIT SUBTRACTION =====
10 - 5 = 5 | Borrow = 0
25 - 12 = 13 | Borrow = 0
100 - 50 = 50 | Borrow = 0
150 - 75 = 75 | Borrow = 0
200 - 25 = 175 | Borrow = 0


In [7]:
pixel_tests = [
    (0, 0),
    (0, 255),
    (255, 0),
    (255, 255),
    (100, 50),
    (50, 100),
    (128, 127),
    (127, 128)
]

In [8]:
print("===== PIXEL ADDITION TEST =====")

for a, b in pixel_tests:

    result, carry = eight_bit_adder_subtractor(
        a,
        b,
        0
    )

    print(
        f"{a:3d} + {b:3d} = "
        f"{result:3d} | Carry = {carry}"
    )

===== PIXEL ADDITION TEST =====
  0 +   0 =   0 | Carry = 0
  0 + 255 = 255 | Carry = 0
255 +   0 = 255 | Carry = 0
255 + 255 = 254 | Carry = 1
100 +  50 = 150 | Carry = 0
 50 + 100 = 150 | Carry = 0
128 + 127 = 255 | Carry = 0
127 + 128 = 255 | Carry = 0


In [9]:
print("===== PIXEL SUBTRACTION TEST =====")

for a, b in pixel_tests:

    result, borrow = eight_bit_adder_subtractor(
        a,
        b,
        1
    )

    print(
        f"{a:3d} - {b:3d} = "
        f"{result:3d} | Borrow = {borrow}"
    )

===== PIXEL SUBTRACTION TEST =====
  0 -   0 =   0 | Borrow = 0
  0 - 255 =   1 | Borrow = 1
255 -   0 = 255 | Borrow = 0
255 - 255 =   0 | Borrow = 0
100 -  50 =  50 | Borrow = 0
 50 - 100 = 206 | Borrow = 1
128 - 127 =   1 | Borrow = 0
127 - 128 = 255 | Borrow = 1


In [10]:
all_tests_passed = True

for a in range(256):

    for b in range(256):

        # Addition
        add_result, add_carry = eight_bit_adder_subtractor(
            a, b, 0
        )

        expected_add = a + b

        if add_result + (add_carry * 256) != expected_add:
            all_tests_passed = False
            print("Addition error:", a, b)

        # Subtraction
        sub_result, sub_borrow = eight_bit_adder_subtractor(
            a, b, 1
        )

        expected_sub = a - b

        if sub_borrow == 0:
            reconstructed_sub = sub_result
        else:
            reconstructed_sub = sub_result - 256

        if reconstructed_sub != expected_sub:
            all_tests_passed = False
            print("Subtraction error:", a, b)

print("===== 8-BIT VERIFICATION =====")

if all_tests_passed:
    print("ALL 65,536 INPUT PAIRS PASSED")
else:
    print("VERIFICATION FAILED")

===== 8-BIT VERIFICATION =====
ALL 65,536 INPUT PAIRS PASSED
